# Amari–Hopfield 联想记忆模型

这一节用二值图像观察联想记忆。每个像素只有两种状态，在计算中编码为 $-1$ 和 $1$。网络不学习类别标签，而是在连接权重中存储若干图像模式。给定受损图像作为初始状态后，网络依次更新各神经元，状态逐步收敛到一个吸引子。

## 1. Hebb 学习规则与异步更新

设第 $\mu$ 个待存模式为 $\boldsymbol{\xi}^{\mu}\in\{-1,1\}^N$。权重由外积相加得到：

$$W=\frac{1}{N}\sum_{\mu}\boldsymbol{\xi}^{\mu}(\boldsymbol{\xi}^{\mu})^\top,\qquad w_{ii}=0.$$

模式恢复时，每一步只随机选择一个神经元：

$$s_i\leftarrow \begin{cases}1,&\sum_jw_{ij}s_j-\theta\ge 0,\\-1,&\text{其他。}\end{cases}$$

对称权重和零对角线使异步更新不会增加能量

$$E(\boldsymbol{s})=-\frac12\boldsymbol{s}^\top W\boldsymbol{s}+\theta\sum_i s_i.$$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from biai.amari_hopfield_model.hopfield_model import (
    OTHER_PATTERN_PATH,
    AmariHopfieldNetwork,
    add_bit_flip_noise,
    add_mask_noise,
    load_patterns,
)

SEED = 0
N_UPDATES = 10_000
patterns = load_patterns()
image1, image2 = patterns[:2]
shared_foreground = np.where((image1 == 1) & (image2 == 1), 1, -1).astype(np.float32)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, image, title in zip(
    axes,
    (image1, image2, shared_foreground),
    ("Pattern 1", "Pattern 2", "Shared foreground"),
    strict=True,
):
    axis.imshow(image.reshape(28, 28), cmap="gray", vmin=-1, vmax=1)
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()

前两幅图像作为存储模式，按照 Hebb 学习规则编码到权重矩阵中。`weight` 必须保持对称且主对角线为零，异步更新的能量才不会增加。

In [ ]:
network = AmariHopfieldNetwork(num_units=patterns.shape[1])
network.store_patterns(patterns)
print("权重是否对称：", np.allclose(network.weight, network.weight.T))
print("主对角线是否为零：", np.allclose(np.diag(network.weight), 0))

## 2. 从像素翻转后的初始状态恢复

独立翻转图像 2 的一部分像素，以得到的二值图像为初始状态执行异步更新。固定的随机种子同时确定翻转位置和更新顺序，运行结果可以复现。

In [ ]:
def show_recovery(original, corrupted, recovered, energy, title):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for axis, image, label in zip(
        axes,
        (original, corrupted, recovered),
        ("Original", "Corrupted", "Recovered"),
        strict=True,
    ):
        axis.imshow(image.reshape(28, 28), cmap="gray", vmin=-1, vmax=1)
        axis.set_title(label)
        axis.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(energy)
    plt.xlabel("Asynchronous update")
    plt.ylabel("Energy")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
rng = np.random.default_rng(SEED)
corrupted_flip = add_bit_flip_noise(image2, probability=0.4, rng=rng)
recovered_flip, energy_flip = network.recover_async(
    corrupted_flip, N_UPDATES, rng=rng, track_energy=True
)
show_recovery(image2, corrupted_flip, recovered_flip, energy_flip, "Bit-flip noise")
print(f"恢复结果与图像 2 的像素匹配率：{np.mean(recovered_flip == image2):.1%}")

## 3. 从局部遮挡后的初始状态恢复

这一次把图像 1 的一个矩形区域统一设为 $1$。未遮挡部分保留足够的模式信息时，网络可以恢复出原来的图像。

In [ ]:
corrupted_mask = add_mask_noise(
    image1.reshape(28, 28),
    row_bounds=(7, 20),
    column_bounds=(8, 25),
).reshape(-1)
recovered_mask, energy_mask = network.recover_async(
    corrupted_mask, N_UPDATES, rng=rng, track_energy=True
)
show_recovery(image1, corrupted_mask, recovered_mask, energy_mask, "Occlusion")
print(f"恢复结果与图像 1 的像素匹配率：{np.mean(recovered_mask == image1):.1%}")

## 4. 不明确的初始状态会收敛到哪个吸引子？

用两幅图像中同时为 $1$ 的像素构造一幅新图像，再随机翻转一半像素。这个初始状态没有对应的目标图像，更新后可能更接近某个存储模式，也可能停在其他稳定状态。

In [ ]:
corrupted_ambiguous = add_bit_flip_noise(shared_foreground, probability=0.5, rng=rng)
recovered_ambiguous, energy_ambiguous = network.recover_async(
    corrupted_ambiguous, N_UPDATES, rng=rng, track_energy=True
)
show_recovery(
    shared_foreground,
    corrupted_ambiguous,
    recovered_ambiguous,
    energy_ambiguous,
    "Ambiguous cue",
)
for index, pattern in enumerate(patterns, start=1):
    match_rate = np.mean(recovered_ambiguous == pattern)
    print(f"与存储模式 {index} 的像素匹配率：{match_rate:.1%}")

## 5. 加载更多示例图像

课程包还提供了 `other_images.npy`，其中有 5 幅二值图像，像素同样用 -1 和 1 编码。先加载并查看它们。

In [ ]:
other_patterns = load_patterns(OTHER_PATTERN_PATH)
fig, axes = plt.subplots(1, len(other_patterns), figsize=(15, 3))
for index, (axis, image) in enumerate(zip(axes, other_patterns, strict=True), start=1):
    axis.imshow(image.reshape(28, 28), cmap="gray", vmin=-1, vmax=1)
    axis.set_title(f"Pattern {index}")
    axis.axis("off")
plt.tight_layout()
plt.show()
print("图像数组形状：", other_patterns.shape)

### 改变存储模式数量

`N_STORED_PATTERNS` 控制从 `other_patterns` 开头取多少幅图像存入网络，默认使用文件中的全部图像；改成较小的整数即可比较不同存储数量。`TARGET_PATTERN_INDEX` 选择用来构造初始状态的存储模式，索引从 0 开始。恢复结果与各存储模式的相似度用像素匹配率衡量。默认设置会收敛到一个不等于任何存储模式的稳定状态，即非记忆吸引子。一次运行只能说明这组设置下模式恢复失败，不能确定网络的存储容量。

In [ ]:
N_STORED_PATTERNS = len(other_patterns)  # Use every image in the file by default.
TARGET_PATTERN_INDEX = 0  # Zero-based index of the target image.
stored_patterns = other_patterns[:N_STORED_PATTERNS]

multi_pattern_network = AmariHopfieldNetwork(num_units=stored_patterns.shape[1])
multi_pattern_network.store_patterns(stored_patterns)
multi_pattern_rng = np.random.default_rng(1020)
multi_pattern_target = stored_patterns[TARGET_PATTERN_INDEX]
multi_pattern_initial_state = add_bit_flip_noise(
    multi_pattern_target, probability=0.2, rng=multi_pattern_rng
)
multi_pattern_recovered, multi_pattern_energy = multi_pattern_network.recover_async(
    multi_pattern_initial_state, N_UPDATES, rng=multi_pattern_rng, track_energy=True
)
pattern_match_rates = np.mean(multi_pattern_recovered == stored_patterns, axis=1)
closest_pattern_index = int(np.argmax(pattern_match_rates))
multi_pattern_is_stored = np.any(np.all(multi_pattern_recovered == stored_patterns, axis=1))
multi_pattern_fixed_point = np.where(
    multi_pattern_network.weight @ multi_pattern_recovered >= multi_pattern_network.threshold,
    1,
    -1,
)
multi_pattern_is_stable = np.array_equal(multi_pattern_recovered, multi_pattern_fixed_point)

show_recovery(
    multi_pattern_target,
    multi_pattern_initial_state,
    multi_pattern_recovered,
    multi_pattern_energy,
    f"{N_STORED_PATTERNS} stored patterns",
)
plt.figure(figsize=(7, 4))
plt.bar(np.arange(1, len(stored_patterns) + 1), pattern_match_rates)
plt.ylim(0, 1)
plt.xlabel("Stored pattern")
plt.ylabel("Pixel match rate")
plt.title("Recovered state compared with stored patterns")
plt.show()
print(f"目标是存储模式 {TARGET_PATTERN_INDEX + 1}")
print(f"恢复结果最接近存储模式 {closest_pattern_index + 1}")
print("恢复结果是否为稳定状态：", multi_pattern_is_stable)
print("恢复结果是否等于某个存储模式：", multi_pattern_is_stored)

## 可选扩展

任选一种改动，先写下预期，再用恢复成功率或像素匹配率检验：

- 逐步提高像素翻转概率，估计吸引域的边界；
- 遍历各个存储模式和多个随机种子，估计模式恢复失败的频率；
- 比较异步更新与同步更新，检查能量是否仍然保持单调不增。